## RealMLP on Predicting Heart Disease

**Score**
- PB:0.95396
- CV:0.9557170083468736

**Note**
- Added CatBoost ALT training cell right after the base CatBoost cell.
- Inserted the Repeated CV comparison cell before the Meta Learning section.
- Replaced the Meta Learning (Ridge) code with the new rank/prob + alpha search version.
- Updated Comparison cell to use new meta outputs.

### Package import

In [1]:
!pip install pytabkit -q

from pathlib import Path
import json
import zipfile
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import warnings
from sklearn.metrics import roc_auc_score
from pytabkit import RealMLP_TD_Classifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

# ---- Config ----
COMP_SLUG = "playground-series-s6e2"
KAGGLE_COMP_DIR = Path("/kaggle/input/competitions/playground-series-s6e2")
KAGGLE_EXT_PATH = Path("/kaggle/input/datasets/neurocipher/heartdisease/Heart_Disease_Prediction.csv")

LOCAL_DATA_DIR = Path("data/raw")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

NEED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 11.5 MB/s eta 0:00:00


In [2]:
def run(cmd: list[str]) -> None:
    p = subprocess.run(cmd, capture_output=True, text=True)
    p.check_returncode()


def ensure_kaggle_cli() -> None:
    try:
        pass
    except Exception:
        subprocess.check_call(["pip", "-q", "install", "kaggle"])


def ensure_kaggle_json_interactive_colab(dst: Path = Path("/content/kaggle.json")) -> Path:
    """
    In Colab: open upload dialog if /content/kaggle.json is missing.
    In non-Colab: just require the file to exist.
    """
    if dst.exists():
        print("Found:", dst)
        return dst

    try:
        from google.colab import files  # type: ignore
    except Exception:
        raise FileNotFoundError(
            f"{dst} not found. Please place kaggle.json at {dst} (Colab) "
            "or provide credentials another way."
        )

    print("Upload your kaggle.json (Kaggle -> Account -> API -> Create New Token)")
    uploaded = files.upload()
    cand = None
    if "kaggle.json" in uploaded:
        cand = "kaggle.json"
    else:
        for name in uploaded.keys():
            if name.endswith(".json"):
                cand = name
                break
    if cand is None:
        raise FileNotFoundError("Upload failed: no .json file received.")

    Path(cand).rename(dst)
    print("Saved to:", dst)
    return dst


def install_kaggle_json(src: Path) -> None:
    """
    Copy /content/kaggle.json -> ~/.kaggle/kaggle.json (chmod 600)
    """
    if not src.exists():
        raise FileNotFoundError(f"{src} not found.")

    dst_dir = Path.home() / ".kaggle"
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / "kaggle.json"

    dst.write_bytes(src.read_bytes())
    try:
        dst.chmod(0o600)
    except Exception:
        pass

    cfg = json.loads(dst.read_text())
    if "username" not in cfg or "key" not in cfg:
        raise ValueError("kaggle.json is missing 'username' or 'key'.")
    print(f"Installed kaggle.json for user: {cfg['username']}")


def local_data_ready(data_dir: Path) -> bool:
    return all((data_dir / f).exists() for f in NEED_FILES)


def download_competition_to(data_dir: Path) -> None:
    """
    Download competition zip(s) and extract into data_dir.
    Assumes kaggle CLI + credentials are ready.
    """
    run(["kaggle", "config", "view"])
    run(["kaggle", "competitions", "download", "-c", COMP_SLUG, "-p", str(data_dir), "--force"])

    zips = list(data_dir.glob("*.zip"))
    if not zips:
        raise FileNotFoundError(f"No zip found in {data_dir} after download.")

    for zp in zips:
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(data_dir)
        print("Unzipped:", zp.name)

    if not local_data_ready(data_dir):
        missing = [f for f in NEED_FILES if not (data_dir / f).exists()]
        raise FileNotFoundError(f"Download/unzip finished but missing: {missing}")


In [3]:
if KAGGLE_COMP_DIR.exists():
    DATA_SRC = "kaggle"
    data_dir = KAGGLE_COMP_DIR
    print("Using Kaggle mounted competition data:", data_dir)
else:
    DATA_SRC = "local"
    data_dir = LOCAL_DATA_DIR
    if local_data_ready(data_dir):
        print("Using local data (already present):", data_dir)
    else:
        print("Local data missing -> download using kaggle.json")
        ensure_kaggle_cli()
        kaggle_json_src = ensure_kaggle_json_interactive_colab(Path("/content/kaggle.json"))
        install_kaggle_json(kaggle_json_src)
        download_competition_to(data_dir)
        print("Download complete -> using local data:", data_dir)


# ---- Load ----
train = pd.read_csv(data_dir / "train.csv")
test  = pd.read_csv(data_dir / "test.csv")
sub   = pd.read_csv(data_dir / "sample_submission.csv")

# external dataset: only available if mounted on Kaggle; optional
original = pd.read_csv(KAGGLE_EXT_PATH) if KAGGLE_EXT_PATH.exists() else None

print("train:", train.shape, "test:", test.shape, "sub:", sub.shape, "original:", None if original is None else original.shape)
print("DATA_SRC:", DATA_SRC)

# External data loading
train_comp = train.copy()

# Encode target with LabelEncoder if not numeric
if not pd.api.types.is_numeric_dtype(train_comp["Heart Disease"]):
    le = LabelEncoder()
    train_comp["Heart Disease"] = le.fit_transform(train_comp["Heart Disease"])
    if original is not None and "Heart Disease" in original.columns:
        original["Heart Disease"] = le.transform(original["Heart Disease"])

# Align external columns to train schema
if original is not None:
    if "Heart Disease" not in original.columns:
        raise ValueError("External dataset missing target column: Heart Disease")

    original_aligned = original.copy()
    # add missing columns
    for col in train_comp.columns:
        if col not in original_aligned.columns:
            original_aligned[col] = np.nan

    # ensure id exists
    if "id" not in original_aligned.columns:
        original_aligned["id"] = -(np.arange(len(original_aligned)) + 1)

    # align column order
    original_aligned = original_aligned[train_comp.columns]

    train_full = pd.concat([train_comp, original_aligned], ignore_index=True)
    train_full["is_external"] = [0] * len(train_comp) + [1] * len(original_aligned)
else:
    train_full = train_comp.copy()
    train_full["is_external"] = 0

# use concatenated data for downstream
train = train_full


Using Kaggle mounted competition data: /kaggle/input/competitions/playground-series-s6e2
train: (630000, 15) test: (270000, 14) sub: (270000, 2) original: None
DATA_SRC: kaggle


In [4]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.set_float32_matmul_precision("high")
N_FOLDS = 5
USE_ALL_CAT = True

print(f"Using device: {DEVICE}")


Using device: cuda


### Data download

In [5]:
display(train.head())
display(test.head())
display(sub.head())

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease,is_external
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,1,0
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,0,0
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,0,0
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,0,0
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,1,0


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3
1,630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3
2,630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7
3,630003,44,0,3,112,177,0,0,168,0,0.9,1,0,3
4,630004,43,1,1,138,267,0,0,163,0,1.8,2,0,7


,id,Heart Disease
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0


In [6]:
# Shapes
print("train:", train.shape)
print("test:", test.shape)

# Column diffs
train_cols = set(train.columns)
test_cols = set(test.columns)
print("Only in train:", sorted(train_cols - test_cols))
print("Only in test:", sorted(test_cols - train_cols))

# dtypes
train.dtypes.to_frame("dtype").head(30)


train: (630000, 16)
test: (270000, 14)
Only in train: ['Heart Disease', 'is_external']
Only in test: []


,dtype
id,int64
Age,int64
Sex,int64
Chest pain type,int64
BP,int64
Cholesterol,int64
FBS over 120,int64
EKG results,int64
Max HR,int64
Exercise angina,int64


### Data Preprocessing

In [7]:
def encode_target_strict(y: pd.Series) -> pd.Series:
    """Map common string labels to {0,1}. Raises if unknown."""
    mapping_candidates = [
        {"No": 0, "Yes": 1},
        {"N": 0, "Y": 1},
        {"Negative": 0, "Positive": 1},
        {"Absent": 0, "Present": 1},
        {"Absence": 0, "Presence": 1},
        {0: 0, 1: 1},
        {"0": 0, "1": 1},
    ]
    uniq = set(pd.Series(y).dropna().unique().tolist())
    for mp in mapping_candidates:
        if uniq.issubset(set(mp.keys())):
            return pd.Series(y).map(mp).astype("int8")
    raise ValueError(f"Unknown target labels: {sorted(list(uniq))}")


# ---- target ----
if not pd.api.types.is_numeric_dtype(train["Heart Disease"]):
    train["Heart Disease"] = encode_target_strict(train["Heart Disease"])
if original is not None and "Heart Disease" in original.columns:
    if not pd.api.types.is_numeric_dtype(original["Heart Disease"]):
        original["Heart Disease"] = encode_target_strict(original["Heart Disease"])

TARGET_COL = "Heart Disease"
ID_COL = "id"
META_COLS = [TARGET_COL, ID_COL, "is_external"]

BASE_FEATURES = [c for c in train.columns if c not in META_COLS]

# Canonical S6E2 semantic categoricals (keep as category for embeddings/encoding)
CANONICAL_CAT = {
    "Sex",
    "Chest pain type",
    "FBS over 120",
    "EKG results",
    "Exercise angina",
    "Slope of ST",
    "Number of vessels fluro",
    "Thallium",
}

def split_cols(df: pd.DataFrame):
    base = [c for c in df.columns if c not in META_COLS]
    cat = [c for c in base if c in CANONICAL_CAT]
    num = [c for c in base if c not in cat]
    return cat, num


def add_external_target_stats(df: pd.DataFrame, original_df: pd.DataFrame | None) -> pd.DataFrame:
    """Merge group-wise target stats from the external/original dataset.
    Safe in Playground comps because original_df labels are not the competition labels.
    """
    if original_df is None:
        return df.copy()

    out = df.copy()
    initial_rows = len(out)

    for col in BASE_FEATURES:
        if col not in original_df.columns:
            continue

        stats = (
            original_df.groupby(col)[TARGET_COL]
            .agg(["mean", "median", "std", "skew", "count"])
            .reset_index()
        )
        stats.columns = [col] + [f"orig_{col}_{s}" for s in ["mean", "median", "std", "skew", "count"]]

        out = out.merge(stats, on=col, how="left")
        if len(out) != initial_rows:
            raise ValueError(f"Merge expanded rows for column {col}! {initial_rows} -> {len(out)}")

        # fill NAs for unseen values
        global_mean = float(original_df[TARGET_COL].mean())
        global_median = float(original_df[TARGET_COL].median())
        fill = {
            f"orig_{col}_mean": global_mean,
            f"orig_{col}_median": global_median,
            f"orig_{col}_std": 0.0,
            f"orig_{col}_skew": 0.0,
            f"orig_{col}_count": 0.0,
        }
        out = out.fillna(value=fill)

    return out



def build_features(
    train_fe: pd.DataFrame,
    test_fe: pd.DataFrame,
    cat_cols: list[str],
    num_cols: list[str],
):
    tr = train_fe.copy()
    te = test_fe.copy()

    # Keep only original + orig_* stats (no periodic/frequency encodings)
    print(f"Train Shape after FE: {tr.shape}")
    print(f"Test Shape after FE:  {te.shape}")

    # ---- Build X/y with clean dtypes ----
    drop_tr = [c for c in META_COLS if c in tr.columns]
    drop_te = [c for c in [ID_COL] if c in te.columns]

    X = tr.drop(columns=drop_tr).copy()
    X_test = te.drop(columns=drop_te).copy()
    y = tr[TARGET_COL].copy()

    cat_cols_final = [c for c in X.columns if c in cat_cols]
    num_cols_final = [c for c in X.columns if c not in cat_cols_final]

    # Ensure no overlap
    assert set(cat_cols_final).isdisjoint(set(num_cols_final))

    for c in cat_cols_final:
        combined = pd.concat([X[c], X_test[c]], axis=0, ignore_index=True)
        cats = pd.Categorical(combined).categories
        X[c] = pd.Categorical(X[c], categories=cats)
        X_test[c] = pd.Categorical(X_test[c], categories=cats)

    for c in num_cols_final:
        X[c] = pd.to_numeric(X[c], errors="coerce").astype("float32")
        X_test[c] = pd.to_numeric(X_test[c], errors="coerce").astype("float32")

    return X, X_test, y, cat_cols_final, num_cols_final


# ---- Feature engineering: match realmlp-ext-target-stats-5-fold-cv ----

# 1) External target stats, same idea as add_engineered_features in the CV notebook
train_fe = add_external_target_stats(train, original)
test_fe  = add_external_target_stats(test, original)

print("Train Shape after FE:", train_fe.shape)
print("Test Shape after FE: ", test_fe.shape)

# 2) Build X, y, X_test just like realmlp-ext-target-stats-5-fold-cv
#    base_features = all columns except target + id
TARGET_COL = "Heart Disease"
ID_COL = "id"

X = train_fe.drop(columns=[ID_COL, TARGET_COL])
y = train_fe[TARGET_COL].astype("int8")
X_test = test_fe.drop(columns=[ID_COL])

# Align columns (train has is_external, test does not)
X = X.drop(columns=["is_external"], errors="ignore")

print("X Shape:", X.shape)
print("X_test Shape:", X_test.shape)

# 3) Optional but close to the CV notebook: treat everything as categorical for RealMLP
#    (In the CV notebook they did astype(str).astype('category') on all columns.)
for col in X.columns:
    X[col] = X[col].astype(str).astype("category")
    X_test[col] = X_test[col].astype(str).astype("category")

# We'll let RealMLP infer categorical columns from dtype, so we don't need cat_cols here.
cat_cols = list(X.columns)


Train Shape after FE: (630000, 16)
Test Shape after FE:  (270000, 14)
X Shape: (630000, 13)
X_test Shape: (270000, 13)


### Data Quality Check

In [8]:
def check_data_quality(df, name="Dataset"):
    print(f"--- Data Quality: {name} ---")
    print(f"Total Rows: {len(df)}")

    cols_to_check = [c for c in df.columns if c != 'id']
    dupes = df.duplicated(subset=cols_to_check).sum()

    nan_counts = df.isnull().sum()
    total_nans = nan_counts.sum()

    print(f"Duplicate Rows (excl. ID): {dupes}")
    print(f"Total NaN values: {total_nans}")
    if total_nans > 0:
        print("\nColumns with NaNs:")
        print(nan_counts[nan_counts > 0])
    print("-" * 30)

check_data_quality(train, "Train")
check_data_quality(test, "Test")

--- Data Quality: Train ---
Total Rows: 630000
Duplicate Rows (excl. ID): 0
Total NaN values: 0
------------------------------
--- Data Quality: Test ---
Total Rows: 270000
Duplicate Rows (excl. ID): 0
Total NaN values: 0
------------------------------


### Feature Uniqueness & Cardinality

In [9]:
# Target distribution analysis

def analyze_uniqueness(df):
    unique_stats = []
    for col in df.columns:
        if col == 'id':
            continue

        n_unique = df[col].nunique()
        dtype = df[col].dtype

        category_guess = "Categorical/Ordinal" if n_unique < 25 else "Continuous"

        if pd.api.types.is_numeric_dtype(df[col]):
            mean_val = float(pd.to_numeric(df[col], errors='coerce').mean())
            std_val = float(pd.to_numeric(df[col], errors='coerce').std())
        else:
            mean_val = float('nan')
            std_val = float('nan')

        unique_stats.append({
            'Feature': col,
            'Unique Values': n_unique,
            'Data Type': dtype,
            'Heuristic Type': category_guess,
            'Mean': mean_val,
            'Std': std_val,
        })

    return pd.DataFrame(unique_stats).sort_values(by='Unique Values')

# class imbalance
comp_mask = (train["is_external"] == 0)
ext_mask  = (train["is_external"] == 1)

print("=== Target distribution: COMP ONLY ===")
print(train.loc[comp_mask, TARGET_COL].value_counts(dropna=False))
print(train.loc[comp_mask, TARGET_COL].value_counts(normalize=True, dropna=False))

print("\n=== Target distribution: EXTERNAL ONLY ===")
print(train.loc[ext_mask, TARGET_COL].value_counts(dropna=False))
print(train.loc[ext_mask, TARGET_COL].value_counts(normalize=True, dropna=False))

print("\n=== Target distribution: MERGED (comp + external) ===")
print(train[TARGET_COL].value_counts(dropna=False))
print(train[TARGET_COL].value_counts(normalize=True, dropna=False))

uniqueness_df = analyze_uniqueness(train)
uniqueness_df


=== Target distribution: COMP ONLY ===
Heart Disease
0    347546
1    282454
Name: count, dtype: int64
Heart Disease
0    0.55166
1    0.44834
Name: proportion, dtype: float64

=== Target distribution: EXTERNAL ONLY ===
Series([], Name: count, dtype: int64)
Series([], Name: proportion, dtype: float64)

=== Target distribution: MERGED (comp + external) ===
Heart Disease
0    347546
1    282454
Name: count, dtype: int64
Heart Disease
0    0.55166
1    0.44834
Name: proportion, dtype: float64


,Feature,Unique Values,Data Type,Heuristic Type,Mean,Std
14,is_external,1,int64,Categorical/Ordinal,0.000000,0.000000
1,Sex,2,int64,Categorical/Ordinal,0.714735,0.451541
8,Exercise angina,2,int64,Categorical/Ordinal,0.273725,0.445870
5,FBS over 120,2,int64,Categorical/Ordinal,0.079987,0.271274
13,Heart Disease,2,int64,Categorical/Ordinal,0.448340,0.497324
10,Slope of ST,3,int64,Categorical/Ordinal,1.455871,0.545192
6,EKG results,3,int64,Categorical/Ordinal,0.981660,0.998783
12,Thallium,3,int64,Categorical/Ordinal,4.618873,1.950007
2,Chest pain type,4,int64,Categorical/Ordinal,3.312752,0.851615
11,Number of vessels fluro,4,int64,Categorical/Ordinal,0.451040,0.798549


### Cross-validation setup


In [10]:
ID_COL = "id"
TARGET_COL = "Heart Disease"

# Competition rows only (is_external == 0)
comp_mask = (train["is_external"] == 0).values

X_comp = X.loc[comp_mask].reset_index(drop=True)
y_comp = y.loc[comp_mask].reset_index(drop=True)
comp_ids = train.loc[comp_mask, ID_COL].reset_index(drop=True)

X_test_comp = X_test.copy()
test_ids = test[ID_COL].reset_index(drop=True)

print("X_comp:", X_comp.shape)
print("y_comp:", y_comp.shape)
print("X_test_comp:", X_test_comp.shape)

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


X_comp: (630000, 13)
y_comp: (630000,)
X_test_comp: (270000, 13)


### Base model and training


In [11]:
!pip install catboost -q


In [12]:
from catboost import CatBoostClassifier, Pool

cat_features_idx = [X_comp.columns.get_loc(c) for c in cat_cols]

oof_cat = np.zeros(len(X_comp), dtype="float32")
test_cat = np.zeros(len(X_test_comp), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"[CatBoost] Fold {fold}/{N_FOLDS}")
    X_tr, X_val = X_comp.iloc[tr_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

    train_pool = Pool(X_tr, label=y_tr, cat_features=cat_features_idx)
    valid_pool = Pool(X_val, label=y_val, cat_features=cat_features_idx)
    test_pool = Pool(X_test_comp, cat_features=cat_features_idx)

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=3.0,
        random_seed=RANDOM_STATE + fold,
        iterations=4000,
        od_type="Iter",
        od_wait=200,
        verbose=200,
        task_type="GPU" if DEVICE == "cuda" else "CPU",
    )

    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    pred_val = model.predict_proba(valid_pool)[:, 1]
    oof_cat[val_idx] = pred_val.astype("float32")

    pred_test = model.predict_proba(test_pool)[:, 1]
    test_cat += (pred_test / N_FOLDS).astype("float32")

cat_oof_auc = roc_auc_score(y_comp, oof_cat)
print(f"[CatBoost] OOF AUC = {cat_oof_auc:.6f}")


[CatBoost] Fold 1/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9406781	best: 0.9406781 (0)	total: 164ms	remaining: 10m 54s
200:	test: 0.9557329	best: 0.9557329 (200)	total: 9.11s	remaining: 2m 52s
400:	test: 0.9559670	best: 0.9559671 (399)	total: 18.2s	remaining: 2m 42s
600:	test: 0.9560046	best: 0.9560046 (599)	total: 27.3s	remaining: 2m 34s
800:	test: 0.9560112	best: 0.9560135 (725)	total: 36.4s	remaining: 2m 25s
1000:	test: 0.9560130	best: 0.9560139 (860)	total: 45.3s	remaining: 2m 15s
bestTest = 0.9560139179
bestIteration = 860
Shrink model to first 861 iterations.
[CatBoost] Fold 2/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9390359	best: 0.9390359 (0)	total: 51.5ms	remaining: 3m 26s
200:	test: 0.9545804	best: 0.9545804 (200)	total: 9.12s	remaining: 2m 52s
400:	test: 0.9548077	best: 0.9548077 (399)	total: 18.2s	remaining: 2m 43s
600:	test: 0.9548554	best: 0.9548554 (600)	total: 27.4s	remaining: 2m 34s
800:	test: 0.9548784	best: 0.9548786 (799)	total: 36.5s	remaining: 2m 25s
1000:	test: 0.9548844	best: 0.9548849 (991)	total: 45.5s	remaining: 2m 16s
1200:	test: 0.9548853	best: 0.9548870 (1023)	total: 54.5s	remaining: 2m 7s
bestTest = 0.9548870027
bestIteration = 1023
Shrink model to first 1024 iterations.
[CatBoost] Fold 3/5


...

0:	test: 0.9394967	best: 0.9394967 (0)	total: 57.2ms	remaining: 3m 48s
200:	test: 0.9553767	best: 0.9553767 (200)	total: 9.02s	remaining: 2m 50s
400:	test: 0.9556374	best: 0.9556378 (399)	total: 18.3s	remaining: 2m 44s
600:	test: 0.9556866	best: 0.9556866 (600)	total: 27.4s	remaining: 2m 35s
800:	test: 0.9557021	best: 0.9557030 (794)	total: 36.5s	remaining: 2m 25s
bestTest = 0.9557030201
bestIteration = 794
Shrink model to first 795 iterations.
[CatBoost] Fold 4/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9393728	best: 0.9393728 (0)	total: 57.5ms	remaining: 3m 50s
200:	test: 0.9549213	best: 0.9549213 (200)	total: 9.08s	remaining: 2m 51s
400:	test: 0.9552022	best: 0.9552024 (397)	total: 18.4s	remaining: 2m 44s
600:	test: 0.9552600	best: 0.9552604 (599)	total: 27.5s	remaining: 2m 35s
800:	test: 0.9552763	best: 0.9552772 (798)	total: 36.6s	remaining: 2m 26s
1000:	test: 0.9552905	best: 0.9552906 (997)	total: 45.7s	remaining: 2m 17s
1200:	test: 0.9552957	best: 0.9552966 (1189)	total: 54.8s	remaining: 2m 7s
1400:	test: 0.9552995	best: 0.9552999 (1318)	total: 1m 4s	remaining: 1m 58s
1600:	test: 0.9553068	best: 0.9553078 (1587)	total: 1m 13s	remaining: 1m 49s
bestTest = 0.9553078413
bestIteration = 1587
Shrink model to first 1588 iterations.
[CatBoost] Fold 5/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9400386	best: 0.9400386 (0)	total: 50.9ms	remaining: 3m 23s
200:	test: 0.9558256	best: 0.9558256 (200)	total: 9.12s	remaining: 2m 52s
400:	test: 0.9560785	best: 0.9560785 (400)	total: 18.2s	remaining: 2m 43s
600:	test: 0.9561218	best: 0.9561218 (600)	total: 27.4s	remaining: 2m 34s
800:	test: 0.9561419	best: 0.9561419 (798)	total: 36.5s	remaining: 2m 25s
1000:	test: 0.9561515	best: 0.9561515 (999)	total: 45.6s	remaining: 2m 16s
1200:	test: 0.9561577	best: 0.9561582 (1185)	total: 54.8s	remaining: 2m 7s
1400:	test: 0.9561605	best: 0.9561611 (1328)	total: 1m 4s	remaining: 1m 58s
1600:	test: 0.9561599	best: 0.9561628 (1463)	total: 1m 13s	remaining: 1m 49s
bestTest = 0.9561628103
bestIteration = 1463
Shrink model to first 1464 iterations.
[CatBoost] OOF AUC = 0.955610


In [13]:
# ---- CatBoost ALT (別設定) ----
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
import numpy as np

cat_features_idx = [X_comp.columns.get_loc(c) for c in cat_cols]

oof_cat_alt = np.zeros(len(X_comp), dtype="float32")
test_cat_alt = np.zeros(len(X_test_comp), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"[CatBoost ALT] Fold {fold}/{N_FOLDS}")

    X_tr, X_val = X_comp.iloc[tr_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

    train_pool = Pool(X_tr, label=y_tr, cat_features=cat_features_idx)
    valid_pool = Pool(X_val, label=y_val, cat_features=cat_features_idx)
    test_pool  = Pool(X_test_comp, cat_features=cat_features_idx)

    model_alt = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=5000,
        learning_rate=0.02,
        depth=5,
        l2_leaf_reg=8.0,
        random_strength=1.5,
        bagging_temperature=1.0,
        od_type="Iter",
        od_wait=250,
        random_seed=RANDOM_STATE + 100 + fold,
        verbose=200,
        task_type="GPU" if DEVICE == "cuda" else "CPU",
    )

    model_alt.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    oof_cat_alt[val_idx] = model_alt.predict_proba(valid_pool)[:, 1].astype("float32")
    test_cat_alt += (model_alt.predict_proba(test_pool)[:, 1] / N_FOLDS).astype("float32")

cat_alt_auc = roc_auc_score(y_comp, oof_cat_alt)
print(f"[CatBoost ALT] OOF AUC = {cat_alt_auc:.6f}")


[CatBoost ALT] Fold 1/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9343147	best: 0.9343147 (0)	total: 46.1ms	remaining: 3m 50s
200:	test: 0.9550491	best: 0.9550491 (200)	total: 7.91s	remaining: 3m 8s
400:	test: 0.9558004	best: 0.9558004 (400)	total: 15.9s	remaining: 3m 2s
600:	test: 0.9559441	best: 0.9559441 (598)	total: 23.9s	remaining: 2m 54s
800:	test: 0.9559984	best: 0.9559985 (799)	total: 31.7s	remaining: 2m 46s
1000:	test: 0.9560163	best: 0.9560168 (992)	total: 39.5s	remaining: 2m 37s
1200:	test: 0.9560301	best: 0.9560301 (1200)	total: 47.4s	remaining: 2m 29s
1400:	test: 0.9560384	best: 0.9560384 (1400)	total: 55.1s	remaining: 2m 21s
1600:	test: 0.9560402	best: 0.9560407 (1589)	total: 1m 2s	remaining: 2m 13s
1800:	test: 0.9560445	best: 0.9560446 (1794)	total: 1m 10s	remaining: 2m 5s
2000:	test: 0.9560468	best: 0.9560475 (1967)	total: 1m 18s	remaining: 1m 57s
2200:	test: 0.9560480	best: 0.9560484 (2173)	total: 1m 25s	remaining: 1m 49s
2400:	test: 0.9560498	best: 0.9560503 (2367)	total: 1m 33s	remaining: 1m 41s
2600:	test: 0.9560525	bes

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9301895	best: 0.9301895 (0)	total: 44.5ms	remaining: 3m 42s
200:	test: 0.9538660	best: 0.9538660 (200)	total: 7.73s	remaining: 3m 4s
400:	test: 0.9546406	best: 0.9546406 (400)	total: 15.6s	remaining: 2m 59s
600:	test: 0.9547859	best: 0.9547859 (600)	total: 23.5s	remaining: 2m 51s
800:	test: 0.9548376	best: 0.9548376 (799)	total: 31.2s	remaining: 2m 43s
1000:	test: 0.9548594	best: 0.9548594 (1000)	total: 38.9s	remaining: 2m 35s
1200:	test: 0.9548741	best: 0.9548742 (1195)	total: 46.6s	remaining: 2m 27s
1400:	test: 0.9548877	best: 0.9548878 (1397)	total: 54.4s	remaining: 2m 19s
1600:	test: 0.9548951	best: 0.9548951 (1598)	total: 1m 2s	remaining: 2m 11s
1800:	test: 0.9549003	best: 0.9549006 (1779)	total: 1m 9s	remaining: 2m 4s
2000:	test: 0.9549037	best: 0.9549041 (1968)	total: 1m 17s	remaining: 1m 56s
2200:	test: 0.9549083	best: 0.9549083 (2188)	total: 1m 25s	remaining: 1m 48s
2400:	test: 0.9549126	best: 0.9549134 (2343)	total: 1m 33s	remaining: 1m 40s
2600:	test: 0.9549133	be

...

0:	test: 0.9298628	best: 0.9298628 (0)	total: 44.1ms	remaining: 3m 40s
200:	test: 0.9545890	best: 0.9545890 (200)	total: 7.74s	remaining: 3m 4s
400:	test: 0.9554281	best: 0.9554281 (400)	total: 15.7s	remaining: 3m
600:	test: 0.9555814	best: 0.9555814 (600)	total: 23.5s	remaining: 2m 52s
800:	test: 0.9556370	best: 0.9556370 (800)	total: 31.3s	remaining: 2m 43s
1000:	test: 0.9556589	best: 0.9556592 (986)	total: 38.9s	remaining: 2m 35s
1200:	test: 0.9556735	best: 0.9556736 (1195)	total: 46.6s	remaining: 2m 27s
1400:	test: 0.9556830	best: 0.9556830 (1400)	total: 54.4s	remaining: 2m 19s
1600:	test: 0.9556892	best: 0.9556894 (1594)	total: 1m 2s	remaining: 2m 11s
1800:	test: 0.9556949	best: 0.9556949 (1795)	total: 1m 9s	remaining: 2m 4s
2000:	test: 0.9556973	best: 0.9556974 (1995)	total: 1m 17s	remaining: 1m 56s
2200:	test: 0.9556984	best: 0.9556995 (2114)	total: 1m 25s	remaining: 1m 48s
2400:	test: 0.9556999	best: 0.9557002 (2381)	total: 1m 33s	remaining: 1m 40s
2600:	test: 0.9557009	best: 0

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9326176	best: 0.9326176 (0)	total: 45.4ms	remaining: 3m 47s
200:	test: 0.9542202	best: 0.9542202 (200)	total: 7.82s	remaining: 3m 6s
400:	test: 0.9550062	best: 0.9550062 (400)	total: 15.7s	remaining: 3m
600:	test: 0.9551830	best: 0.9551830 (600)	total: 23.6s	remaining: 2m 52s
800:	test: 0.9552441	best: 0.9552441 (799)	total: 31.4s	remaining: 2m 44s
1000:	test: 0.9552717	best: 0.9552717 (988)	total: 39.1s	remaining: 2m 36s
1200:	test: 0.9552867	best: 0.9552870 (1198)	total: 46.9s	remaining: 2m 28s
1400:	test: 0.9553009	best: 0.9553009 (1400)	total: 54.6s	remaining: 2m 20s
1600:	test: 0.9553096	best: 0.9553096 (1599)	total: 1m 2s	remaining: 2m 12s
1800:	test: 0.9553156	best: 0.9553158 (1786)	total: 1m 10s	remaining: 2m 4s
2000:	test: 0.9553176	best: 0.9553180 (1992)	total: 1m 18s	remaining: 1m 56s
2200:	test: 0.9553225	best: 0.9553226 (2198)	total: 1m 25s	remaining: 1m 49s
2400:	test: 0.9553262	best: 0.9553263 (2395)	total: 1m 33s	remaining: 1m 41s
2600:	test: 0.9553292	best: 

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9336584	best: 0.9336584 (0)	total: 45.1ms	remaining: 3m 45s
200:	test: 0.9551257	best: 0.9551257 (200)	total: 7.85s	remaining: 3m 7s
400:	test: 0.9559013	best: 0.9559013 (400)	total: 15.9s	remaining: 3m 2s
600:	test: 0.9560687	best: 0.9560687 (600)	total: 23.7s	remaining: 2m 53s
800:	test: 0.9561200	best: 0.9561200 (800)	total: 31.6s	remaining: 2m 45s
1000:	test: 0.9561435	best: 0.9561435 (1000)	total: 39.4s	remaining: 2m 37s
1200:	test: 0.9561573	best: 0.9561573 (1195)	total: 47.1s	remaining: 2m 29s
1400:	test: 0.9561674	best: 0.9561674 (1400)	total: 55s	remaining: 2m 21s
1600:	test: 0.9561737	best: 0.9561737 (1600)	total: 1m 2s	remaining: 2m 13s
1800:	test: 0.9561779	best: 0.9561781 (1796)	total: 1m 10s	remaining: 2m 5s
2000:	test: 0.9561787	best: 0.9561787 (1997)	total: 1m 18s	remaining: 1m 57s
2200:	test: 0.9561821	best: 0.9561821 (2200)	total: 1m 26s	remaining: 1m 49s
2400:	test: 0.9561841	best: 0.9561845 (2337)	total: 1m 34s	remaining: 1m 42s
2600:	test: 0.9561849	best

In [14]:
param_grid = {
        # 'device': 'cuda',
        'random_state': 42,
        'verbosity': 2,
        'n_epochs': 100,
        'batch_size': 2**12,
        'n_ens': 8,
        'use_early_stopping': True,
        'early_stopping_additive_patience': 20,
        'early_stopping_multiplicative_patience': 1,
        'act': "mish",
        'embedding_size': 8,
        'first_layer_lr_factor': 0.5962121993798933,
        'val_metric_name': '1-auc_ovr',
        'hidden_sizes': "rectangular",
        'hidden_width': 384,
        'lr': 0.04,
        'ls_eps': 0.011498317194338772,
        'ls_eps_sched': "coslog4",
        'max_one_hot_cat_size': 18,
        'n_hidden_layers': 4,
        'p_drop': 0.07301419697186451,
        'p_drop_sched': "flat_cos",
        'plr_hidden_1': 16,
        'plr_hidden_2': 8,
        'plr_lr_factor': 0.1151437622270563,
        'plr_sigma': 2.3316811282666916,
        'scale_lr_factor': 2.244801835541429,
        'sq_mom': 1.0 - 0.011834054955582318,
        'wd': 0.02369230879235962,
    }
param_grid['device'] = DEVICE


def make_model(param_grid, cat_cols):
    try:
        return RealMLP_TD_Classifier(**param_grid, cat_col_names=cat_cols)
    except TypeError:
        return RealMLP_TD_Classifier(**param_grid)

In [15]:
test_realmlp = np.zeros(len(X_test_comp), dtype="float32")
oof_realmlp = np.zeros(len(X_comp), dtype="float32")
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"\n--- RealMLP Fold {fold}/{N_FOLDS} ---")

    X_tr, X_val = X_comp.iloc[train_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[train_idx], y_comp.iloc[val_idx]

    
    model = make_model(param_grid, cat_cols)
    model.fit(X_tr, y_tr.values, X_val, y_val.values)

    val_probs = model.predict_proba(X_val)[:, 1]
    test_probs = model.predict_proba(X_test_comp)[:, 1]

    # OOF を fold ごとに埋める
    oof_realmlp[val_idx] = val_probs.astype("float32")

    # test は fold 平均
    test_realmlp += (test_probs / N_FOLDS).astype("float32")

    score = roc_auc_score(y_val, val_probs)
    fold_scores.append(score)
    print(f"Fold {fold} ROC-AUC: {score:.6f}")

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print(f"\n[RealMLP] OOF AUC = {roc_auc_score(y_comp, oof_realmlp):.6f}")
print("Fold scores:", fold_scores)


--- RealMLP Fold 1/5 ---
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/100: val 1-auc_ovr = 0.047396
Epoch 2/100: val 1-auc_ovr = 0.044425
Epoch 3/100: val 1-auc_ovr = 0.043934
Epoch 4/100: val 1-auc_ovr = 0.043966
Epoch 5/100: val 1-auc_ovr = 0.043908
Epoch 6/100: val 1-auc_ovr = 0.043883
Epoch 7/100: val 1-auc_ovr = 0.043881
Epoch 8/100: val 1-auc_ovr = 0.043909
Epoch 9/100: val 1-auc_ovr = 0.043943
Epoch 10/100: val 1-auc_ovr = 0.043965
Epoch 11/100: val 1-auc_ovr = 0.043936
Epoch 12/100: val 1-auc_ovr = 0.043983
Epoch 13/100: val 1-auc_ovr = 0.043933
Epoch 14/100: val 1-auc_ovr = 0.043955
Epoch 15/100: val 1-auc_ovr = 0.043933
Epoch 16/100: val 1-auc_ovr = 0.043922
Epoch 17/100: val 1-auc_ovr = 0.043947
Epoch 18/100: val 1-auc_ovr = 0.043961
Epoch 19/100: val 1-auc_ovr = 0.043973
Epoch 20/100: val 1-auc_ovr = 0.043979
Epoch 21/100: val 1-auc_ovr = 0.043983
Epoch 22/100: val 1-auc_ovr = 0.043984
Epoch 23/100: val 1-auc_ovr = 0.044007
Epoch 24/100: val 1-auc_ovr = 0.044003
Epoch 25/100: val 1-auc_ovr = 0.043981
Epoch 26/100: val 1-auc_ovr = 0.04

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 1 ROC-AUC: 0.956119

--- RealMLP Fold 2/5 ---
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


...

Fold 4 ROC-AUC: 0.955436

--- RealMLP Fold 5/5 ---
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/100: val 1-auc_ovr = 0.047628
Epoch 2/100: val 1-auc_ovr = 0.044322
Epoch 3/100: val 1-auc_ovr = 0.043898
Epoch 4/100: val 1-auc_ovr = 0.043818
Epoch 5/100: val 1-auc_ovr = 0.043801
Epoch 6/100: val 1-auc_ovr = 0.043785
Epoch 7/100: val 1-auc_ovr = 0.043774
Epoch 8/100: val 1-auc_ovr = 0.043792
Epoch 9/100: val 1-auc_ovr = 0.043812
Epoch 10/100: val 1-auc_ovr = 0.043804
Epoch 11/100: val 1-auc_ovr = 0.043865
Epoch 12/100: val 1-auc_ovr = 0.043801
Epoch 13/100: val 1-auc_ovr = 0.043811
Epoch 14/100: val 1-auc_ovr = 0.043784
Epoch 15/100: val 1-auc_ovr = 0.043829
Epoch 16/100: val 1-auc_ovr = 0.043795
Epoch 17/100: val 1-auc_ovr = 0.043799
Epoch 18/100: val 1-auc_ovr = 0.043831
Epoch 19/100: val 1-auc_ovr = 0.043840
Epoch 20/100: val 1-auc_ovr = 0.043843
Epoch 21/100: val 1-auc_ovr = 0.043851
Epoch 22/100: val 1-auc_ovr = 0.043857
Epoch 23/100: val 1-auc_ovr = 0.043858
Epoch 24/100: val 1-auc_ovr = 0.043841
Epoch 25/100: val 1-auc_ovr = 0.043835
Epoch 26/100: val 1-auc_ovr = 0.04

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 5 ROC-AUC: 0.956226

[RealMLP] OOF AUC = 0.955657
Fold scores: [np.float64(0.9561193713407058), np.float64(0.9549367554466837), np.float64(0.9557678580904292), np.float64(0.9554355320169413), np.float64(0.9562259479202457)]


### Rename / Noise Feature Fallback


In [16]:
# 既存変数名の整理（必要なら）
oof_real = oof_realmlp.copy()
test_real = test_realmlp.copy()


In [17]:
import numpy as np
import pandas as pd

def safe_array(x, n, fill_value=0.0, name="arr"):
    """x が None / 未定義でも長さ n の配列を返す"""
    if x is None:
        return np.full(n, fill_value, dtype="float64")
    arr = np.asarray(x, dtype="float64")
    if len(arr) != n:
        raise ValueError(f"{name} length mismatch: expected {n}, got {len(arr)}")
    return arr

# 例: oof_p_noise, test_p_noise がある notebook ならそのまま使う
# ない notebook なら 0 埋め
try:
    _oof_p_noise = oof_p_noise
except NameError:
    _oof_p_noise = None

try:
    _test_p_noise = test_p_noise
except NameError:
    _test_p_noise = None

oof_p_noise_feat = safe_array(_oof_p_noise, len(y_comp), fill_value=0.0, name="oof_p_noise")
test_p_noise_feat = safe_array(_test_p_noise, len(X_test_comp), fill_value=0.0, name="test_p_noise")

print("oof_p_noise_feat:", pd.Series(oof_p_noise_feat).describe())
print("test_p_noise_feat:", pd.Series(test_p_noise_feat).describe())


oof_p_noise_feat: count    630000.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
dtype: float64
test_p_noise_feat: count    270000.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
dtype: float64


### XGBoost (Diverse 3rd Model)


In [18]:
!pip install xgboost -q
import xgboost as xgb
from sklearn.metrics import roc_auc_score

def to_xgb_matrix(X_df: pd.DataFrame, X_test_df: pd.DataFrame, cat_cols: list[str]):
    """
    category列を codes 化して XGBoost 用の数値行列を作る。
    train/test でカテゴリ対応を揃える。
    """
    X_tr = X_df.copy()
    X_te = X_test_df.copy()

    for c in cat_cols:
        # train/test 結合して categories を揃える
        combined = pd.concat([X_tr[c], X_te[c]], axis=0, ignore_index=True)
        cats = pd.Categorical(combined).categories

        X_tr[c] = pd.Categorical(X_tr[c], categories=cats).codes.astype("int32")
        X_te[c] = pd.Categorical(X_te[c], categories=cats).codes.astype("int32")

    # 念のため全列数値化
    for c in X_tr.columns:
        X_tr[c] = pd.to_numeric(X_tr[c], errors="coerce").astype("float32")
        X_te[c] = pd.to_numeric(X_te[c], errors="coerce").astype("float32")

    return X_tr, X_te

X_comp_xgb, X_test_xgb = to_xgb_matrix(X_comp, X_test_comp, cat_cols)
print(X_comp_xgb.shape, X_test_xgb.shape)


(630000, 13) (270000, 13)


In [19]:
oof_xgb = np.zeros(len(X_comp_xgb), dtype="float32")
test_xgb = np.zeros(len(X_test_xgb), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp_xgb, y_comp), start=1):
    print(f"[XGB] Fold {fold}/{N_FOLDS}")
    X_tr, X_val = X_comp_xgb.iloc[tr_idx], X_comp_xgb.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

    # Catと少し違う寄りの設定
    model = xgb.XGBClassifier(
        n_estimators=5000,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=8,
        subsample=0.75,
        colsample_bytree=0.75,
        reg_alpha=0.5,
        reg_lambda=2.0,
        gamma=0.5,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=RANDOM_STATE + fold,
        n_jobs=-1,
        device="gpu" if DEVICE == "cuda" else "cpu",
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=200,
    )

    pred_val = model.predict_proba(X_val)[:, 1]
    pred_test = model.predict_proba(X_test_xgb)[:, 1]

    oof_xgb[val_idx] = pred_val.astype("float32")
    test_xgb += (pred_test / N_FOLDS).astype("float32")

xgb_oof_auc = roc_auc_score(y_comp, oof_xgb)
print(f"[XGB] OOF AUC = {xgb_oof_auc:.6f}")


[XGB] Fold 1/5
[0]	validation_0-auc:0.91974
[200]	validation_0-auc:0.95400
[400]	validation_0-auc:0.95485
[600]	validation_0-auc:0.95530
[800]	validation_0-auc:0.95556
[1000]	validation_0-auc:0.95570
[1200]	validation_0-auc:0.95576
[1400]	validation_0-auc:0.95580
[1600]	validation_0-auc:0.95583
[1800]	validation_0-auc:0.95583
[2000]	validation_0-auc:0.95583
[2200]	validation_0-auc:0.95582
[2400]	validation_0-auc:0.95581
[2600]	validation_0-auc:0.95579
[2800]	validation_0-auc:0.95578
[3000]	validation_0-auc:0.95577
[3200]	validation_0-auc:0.95576
[3400]	validation_0-auc:0.95574
[3600]	validation_0-auc:0.95572
[3800]	validation_0-auc:0.95570
[4000]	validation_0-auc:0.95569
[4200]	validation_0-auc:0.95567
[4400]	validation_0-auc:0.95565
[4600]	validation_0-auc:0.95563
[4800]	validation_0-auc:0.95561
[4999]	validation_0-auc:0.95559
[XGB] Fold 2/5
[0]	validation_0-auc:0.89475
[200]	validation_0-auc:0.95295
[400]	validation_0-auc:0.95395
[600]	validation_0-auc:0.95435
[800]	validation_0-auc:

In [20]:

# SEEDS = [42, 2024, 2025]

# test_realmlp = np.zeros(len(X_test_comp), dtype="float32")
# oof_realmlp = np.zeros(len(X_comp), dtype="float32")

# for seed in SEEDS:
#     print(f"\n==== SEED {seed} ====")
#     param_grid['random_state'] = seed

#     oof_realmlp = np.zeros(len(X_comp), dtype="float32")
#     test_realmlp = np.zeros(len(X_test_comp), dtype="float32")
#     fold_scores = []

#     for fold, (train_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
#         print(f"\n--- RealMLP Fold {fold}/{N_FOLDS} ---")

#         X_tr, X_val = X_comp.iloc[train_idx], X_comp.iloc[val_idx]
#         y_tr, y_val = y_comp.iloc[train_idx], y_comp.iloc[val_idx]

#         model = make_model(param_grid, cat_cols)
#         model.fit(X_tr, y_tr.values, X_val, y_val.values)

#         val_probs = model.predict_proba(X_val)[:, 1]
#         test_probs = model.predict_proba(X_test_comp)[:, 1]

#         oof_realmlp[val_idx] = val_probs.astype("float32")
#         test_realmlp += (test_probs / N_FOLDS).astype("float32")

#         score = roc_auc_score(y_val, val_probs)
#         fold_scores.append(score)
#         print(f"Fold {fold} ROC-AUC: {score:.6f}")

#         if DEVICE == 'cuda':
#             torch.cuda.empty_cache()

#     print(f"\n[RealMLP][seed={seed}] OOF AUC = {roc_auc_score(y_comp, oof_realmlp):.6f}")
#     print("Fold scores:", fold_scores)

#     oof_realmlp += oof_realmlp / len(SEEDS)
#     test_realmlp += test_realmlp / len(SEEDS)

# print(f"\n[RealMLP][ensemble] OOF AUC = {roc_auc_score(y_comp, oof_realmlp):.6f}")


### Evaluation

In [21]:
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

def rank01(x: np.ndarray) -> np.ndarray:
    """0-1 rank (tiesは単純argsort版; 必要ならrankdataに差し替え可)"""
    x = np.asarray(x, dtype="float64")
    order = np.argsort(x)
    ranks = np.empty_like(order, dtype="float64")
    ranks[order] = np.arange(len(x), dtype="float64")
    return (ranks + 1.0) / (len(x) + 2.0)

def minmax01(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype="float64")
    mn, mx = x.min(), x.max()
    if mx - mn < 1e-12:
        return np.full_like(x, 0.5)
    return (x - mn) / (mx - mn)

# Base OOF AUC
print("[Base] Cat     OOF AUC =", roc_auc_score(y_comp, oof_cat))
print("[Base] RealMLP OOF AUC =", roc_auc_score(y_comp, oof_real))
if "oof_xgb" in globals():
    print("[Base] XGB     OOF AUC =", roc_auc_score(y_comp, oof_xgb))

print("Spearman(cat, real) =", spearmanr(oof_cat, oof_real).correlation)
if "oof_xgb" in globals():
    print("Spearman(cat, xgb)  =", spearmanr(oof_cat, oof_xgb).correlation)
    print("Spearman(real, xgb) =", spearmanr(oof_real, oof_xgb).correlation)


[Base] Cat     OOF AUC = 0.9556097156693959
[Base] RealMLP OOF AUC = 0.9556570025961212
[Base] XGB     OOF AUC = 0.9552204345460825
Spearman(cat, real) = 0.9980183425411407
Spearman(cat, xgb)  = 0.9972195919089082
Spearman(real, xgb) = 0.9971546813458525


### Meta Features


In [22]:
# ---- 使用するベースモデルを定義 ----
base_oof = {
    "cat": np.asarray(oof_cat, dtype="float64"),
    "real": np.asarray(oof_real, dtype="float64"),
}
base_test = {
    "cat": np.asarray(test_cat, dtype="float64"),
    "real": np.asarray(test_real, dtype="float64"),
}

# 3本目を追加（XGB or Cat_ALT）
if "oof_xgb" in globals():
    base_oof["xgb"] = np.asarray(oof_xgb, dtype="float64")
    base_test["xgb"] = np.asarray(test_xgb, dtype="float64")
elif "oof_cat_alt" in globals():
    base_oof["cat_alt"] = np.asarray(oof_cat_alt, dtype="float64")
    base_test["cat_alt"] = np.asarray(test_cat_alt, dtype="float64")

model_names = list(base_oof.keys())
print("Use models for meta-stacking:", model_names)

# ---- rank特徴・prob特徴を作る ----
meta_oof_df = pd.DataFrame(index=np.arange(len(y_comp)))
meta_test_df = pd.DataFrame(index=np.arange(len(X_test_comp)))

for name in model_names:
    p_oof = base_oof[name]
    p_te = base_test[name]

    meta_oof_df[f"prob_{name}"] = p_oof
    meta_test_df[f"prob_{name}"] = p_te

    meta_oof_df[f"rank_{name}"] = rank01(p_oof)
    meta_test_df[f"rank_{name}"] = rank01(p_te)

# ---- pairwise gap特徴（順位差・確率差）----
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        a = model_names[i]
        b = model_names[j]

        meta_oof_df[f"rank_gap_{a}_{b}"] = np.abs(meta_oof_df[f"rank_{a}"] - meta_oof_df[f"rank_{b}"])
        meta_test_df[f"rank_gap_{a}_{b}"] = np.abs(meta_test_df[f"rank_{a}"] - meta_test_df[f"rank_{b}"])

        meta_oof_df[f"prob_gap_{a}_{b}"] = np.abs(meta_oof_df[f"prob_{a}"] - meta_oof_df[f"prob_{b}"])
        meta_test_df[f"prob_gap_{a}_{b}"] = np.abs(meta_test_df[f"prob_{a}"] - meta_test_df[f"prob_{b}"])

# ---- 補助特徴としての p_noise ----
meta_oof_df["p_noise"] = oof_p_noise_feat
meta_test_df["p_noise"] = test_p_noise_feat

# ---- 参考: ベース平均（補助列）----
prob_cols_oof = [f"prob_{m}" for m in model_names]
rank_cols_oof = [f"rank_{m}" for m in model_names]

meta_oof_df["prob_mean"] = meta_oof_df[prob_cols_oof].mean(axis=1)
meta_test_df["prob_mean"] = meta_test_df[[f"prob_{m}" for m in model_names]].mean(axis=1)

meta_oof_df["rank_mean"] = meta_oof_df[rank_cols_oof].mean(axis=1)
meta_test_df["rank_mean"] = meta_test_df[[f"rank_{m}" for m in model_names]].mean(axis=1)

print(meta_oof_df.shape, meta_test_df.shape)
display(meta_oof_df.head())


Use models for meta-stacking: ['cat', 'real', 'xgb']
(630000, 15) (270000, 15)


,prob_cat,rank_cat,prob_real,rank_real,prob_xgb,rank_xgb,rank_gap_cat_real,prob_gap_cat_real,rank_gap_cat_xgb,prob_gap_cat_xgb,rank_gap_real_xgb,prob_gap_real_xgb,p_noise,prob_mean,rank_mean
0,0.997369,0.972416,0.993484,0.939211,0.997464,0.971899,0.033205,0.003886,0.000517,0.000094,0.032687,0.003980,0.0,0.996105,0.961175
1,0.010252,0.103195,0.008669,0.076784,0.007580,0.070173,0.026411,0.001583,0.033022,0.002673,0.006611,0.001090,0.0,0.008834,0.083384
2,0.008678,0.087098,0.009887,0.089938,0.011512,0.109493,0.002840,0.001209,0.022395,0.002834,0.019555,0.001625,0.0,0.010025,0.095510
3,0.048937,0.290005,0.048276,0.287318,0.041397,0.269317,0.002687,0.000661,0.020689,0.007540,0.018002,0.006880,0.0,0.046203,0.282213
4,0.996724,0.964676,0.997424,0.980106,0.998857,0.988584,0.015430,0.000700,0.023908,0.002132,0.008478,0.001432,0.0,0.997669,0.977789


In [23]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

# ========= Utils =========
def rank01(x):
    x = np.asarray(x, dtype=np.float64)
    r = rankdata(x, method="average")  # tiesに少し丁寧
    return (r - 0.5) / len(r)

def minmax01(x):
    x = np.asarray(x, dtype=np.float64)
    mn, mx = x.min(), x.max()
    if mx - mn < 1e-12:
        return np.full_like(x, 0.5)
    return (x - mn) / (mx - mn)

def build_meta_features(pred_dict, feature_mode="rank+prob"):
    """
    pred_dict: {"cat": oof_cat, "real": oof_real, ...}
    feature_mode: "rank", "prob", "rank+prob"
    """
    cols = []
    names = []

    if feature_mode in ("rank", "rank+prob"):
        for k, v in pred_dict.items():
            cols.append(rank01(v))
            names.append(f"{k}_rank")

    if feature_mode in ("prob", "rank+prob"):
        for k, v in pred_dict.items():
            cols.append(np.asarray(v, dtype=np.float64))
            names.append(f"{k}_prob")

    X_meta = np.column_stack(cols)
    return X_meta, names

def repeated_cv_compare(
    y,
    pred_dict,
    seeds=(42, 2024, 2025),
    n_splits=5,
    ridge_alphas=(0.01, 0.1, 1.0, 3.0, 10.0, 30.0, 100.0),
    feature_mode="rank+prob",
    use_positive_clip=False,
):
    """
    比較対象:
      - Cat only
      - Rank mean (利用可能モデルのrank平均)
      - Meta Ridge (OOF上で further CV)
    """
    y = np.asarray(y, dtype=np.int64)

    # available models
    model_order = [k for k in ["cat", "real", "xgb", "cat_alt"] if k in pred_dict]
    pred_dict = {k: np.asarray(pred_dict[k], dtype=np.float64) for k in model_order}

    # 固定ベースライン（全体OOFでの参考）
    cat_auc_full = roc_auc_score(y, pred_dict["cat"])
    rank_mean_full = np.mean([rank01(pred_dict[k]) for k in model_order], axis=0)
    rank_mean_auc_full = roc_auc_score(y, rank_mean_full)

    print("=== Full OOF reference ===")
    print(f"[Cat only]   {cat_auc_full:.6f}")
    print(f"[Rank mean]  {rank_mean_auc_full:.6f}")
    print("models:", model_order)

    rows = []

    for seed in seeds:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

        # ---- Cat only (same pred reused, but fold-wise scoreを見る) ----
        cat_fold_scores = []
        rank_mean_fold_scores = []

        # ---- Meta Ridge OOF (further CV on OOF predictions) ----
        meta_oof_pred_best = np.zeros(len(y), dtype=np.float64)
        best_alpha_each_fold = []

        for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
            y_tr, y_va = y[tr_idx], y[va_idx]

            # fold score for cat-only
            cat_fold_scores.append(roc_auc_score(y_va, pred_dict["cat"][va_idx]))

            # fold score for rank-mean
            rank_mean_va = np.mean([rank01(pred_dict[k])[va_idx] for k in model_order], axis=0)
            rank_mean_fold_scores.append(roc_auc_score(y_va, rank_mean_va))

            # meta features
            X_tr, feat_names = build_meta_features({k: pred_dict[k][tr_idx] for k in model_order}, feature_mode)
            X_va, _ = build_meta_features({k: pred_dict[k][va_idx] for k in model_order}, feature_mode)

            # alpha探索（このfold内でtr→vaで選ぶ簡易版）
            # ※より厳密にやるなら inner-CV。まずは軽量版で十分。
            best_auc = -1
            best_alpha = None
            best_pred = None

            for alpha in ridge_alphas:
                model = Ridge(alpha=alpha, fit_intercept=True, random_state=None)
                model.fit(X_tr, y_tr)
                pred_va = model.predict(X_va)

                # AUCは単調変換不変。clip不要だが、見やすさで任意。
                if use_positive_clip:
                    pred_eval = minmax01(pred_va)
                else:
                    pred_eval = pred_va

                auc = roc_auc_score(y_va, pred_eval)

                if auc > best_auc:
                    best_auc = auc
                    best_alpha = alpha
                    best_pred = pred_va

            meta_oof_pred_best[va_idx] = best_pred
            best_alpha_each_fold.append(best_alpha)

        meta_auc = roc_auc_score(y, meta_oof_pred_best)

        rows.append({
            "seed": seed,
            "cat_auc_mean_fold": float(np.mean(cat_fold_scores)),
            "rank_mean_auc_mean_fold": float(np.mean(rank_mean_fold_scores)),
            "meta_ridge_auc": float(meta_auc),
            "meta_alpha_mode": pd.Series(best_alpha_each_fold).mode().iloc[0],
            "meta_alpha_list": best_alpha_each_fold,
        })

        print(f"\n[seed={seed}]")
        print(f"  Cat mean-fold AUC   = {np.mean(cat_fold_scores):.6f}")
        print(f"  Rank mean mean-fold = {np.mean(rank_mean_fold_scores):.6f}")
        print(f"  Meta Ridge OOF AUC  = {meta_auc:.6f}")
        print(f"  best alpha/folds    = {best_alpha_each_fold}")

    df_res = pd.DataFrame(rows)

    print("\n=== Repeated CV summary (3 seeds) ===")
    for col in ["cat_auc_mean_fold", "rank_mean_auc_mean_fold", "meta_ridge_auc"]:
        m = df_res[col].mean()
        s = df_res[col].std(ddof=1) if len(df_res) > 1 else 0.0
        print(f"{col:24s}: {m:.6f} ± {s:.6f}")

    return df_res

pred_dict = {
    "cat": oof_cat,
    "real": oof_real,
}
if "oof_xgb" in globals():
    pred_dict["xgb"] = oof_xgb
if "oof_cat_alt" in globals():
    pred_dict["cat_alt"] = oof_cat_alt

df_cv_cmp = repeated_cv_compare(
    y=y_comp.values,
    pred_dict=pred_dict,
    seeds=(42, 2024, 2025),
    n_splits=5,
    ridge_alphas=(0.01, 0.1, 1.0, 3.0, 10.0, 30.0, 100.0),
    feature_mode="rank+prob",
    use_positive_clip=False,
)

display(df_cv_cmp)


=== Full OOF reference ===
[Cat only]   0.955610
[Rank mean]  0.955694
models: ['cat', 'real', 'xgb', 'cat_alt']

[seed=42]
  Cat mean-fold AUC   = 0.955615
  Rank mean mean-fold = 0.955697
  Meta Ridge OOF AUC  = 0.955719
  best alpha/folds    = [30.0, 3.0, 30.0, 10.0, 10.0]

[seed=2024]
  Cat mean-fold AUC   = 0.955609
  Rank mean mean-fold = 0.955693
  Meta Ridge OOF AUC  = 0.955723
  best alpha/folds    = [10.0, 0.01, 3.0, 3.0, 10.0]

[seed=2025]
  Cat mean-fold AUC   = 0.955609
  Rank mean mean-fold = 0.955694
  Meta Ridge OOF AUC  = 0.955723
  best alpha/folds    = [3.0, 3.0, 10.0, 1.0, 0.01]

=== Repeated CV summary (3 seeds) ===
cat_auc_mean_fold       : 0.955611 ± 0.000003
rank_mean_auc_mean_fold : 0.955695 ± 0.000002
meta_ridge_auc          : 0.955722 ± 0.000002


,seed,cat_auc_mean_fold,rank_mean_auc_mean_fold,meta_ridge_auc,meta_alpha_mode,meta_alpha_list
0,42,0.955615,0.955697,0.955719,10.0,"[30.0, 3.0, 30.0, 10.0, 10.0]"
1,2024,0.955609,0.955693,0.955723,3.0,"[10.0, 0.01, 3.0, 3.0, 10.0]"
2,2025,0.955609,0.955694,0.955723,3.0,"[3.0, 3.0, 10.0, 1.0, 0.01]"


### Meta Learning (Ridge)


In [24]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
import numpy as np
import pandas as pd

def rank01(x):
    x = np.asarray(x, dtype=np.float64)
    r = rankdata(x, method="average")
    return (r - 0.5) / len(r)

def build_meta_matrix_from_arrays(oof_dict, test_dict, model_order, feature_mode="rank+prob"):
    Xo_cols, Xt_cols, names = [], [], []

    if feature_mode in ("rank", "rank+prob"):
        for m in model_order:
            Xo_cols.append(rank01(oof_dict[m]))
            Xt_cols.append(rank01(test_dict[m]))
            names.append(f"{m}_rank")

    if feature_mode in ("prob", "rank+prob"):
        for m in model_order:
            Xo_cols.append(np.asarray(oof_dict[m], dtype=np.float64))
            Xt_cols.append(np.asarray(test_dict[m], dtype=np.float64))
            names.append(f"{m}_prob")

    X_meta_oof = np.column_stack(Xo_cols)
    X_meta_test = np.column_stack(Xt_cols)
    return X_meta_oof, X_meta_test, names

def fit_meta_ridge_oof_and_test(
    y,
    oof_dict,
    test_dict,
    model_order,
    alpha_grid=(0.01, 0.1, 1.0, 3.0, 10.0, 30.0, 100.0),
    feature_mode="rank+prob",
    n_splits=5,
    seed=42,
):
    y = np.asarray(y, dtype=np.int64)
    X_meta_oof, X_meta_test, feat_names = build_meta_matrix_from_arrays(
        oof_dict=oof_dict,
        test_dict=test_dict,
        model_order=model_order,
        feature_mode=feature_mode,
    )

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    meta_oof_pred = np.zeros(len(y), dtype=np.float64)
    meta_test_pred_folds = []
    fold_info = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_meta_oof, y), start=1):
        X_tr, X_va = X_meta_oof[tr_idx], X_meta_oof[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        best_auc = -1.0
        best_alpha = None
        best_model = None
        best_va_pred = None

        for alpha in alpha_grid:
            model = Ridge(alpha=alpha, fit_intercept=True)
            model.fit(X_tr, y_tr)
            va_pred = model.predict(X_va)
            auc = roc_auc_score(y_va, va_pred)

            if auc > best_auc:
                best_auc = auc
                best_alpha = alpha
                best_model = model
                best_va_pred = va_pred

        meta_oof_pred[va_idx] = best_va_pred
        meta_test_pred_folds.append(best_model.predict(X_meta_test))
        fold_info.append({
            "fold": fold,
            "best_alpha": best_alpha,
            "val_auc": best_auc,
        })

        print(f"[Meta Ridge][Fold {fold}] AUC={best_auc:.6f}, alpha={best_alpha}")

    meta_oof_auc = roc_auc_score(y, meta_oof_pred)
    meta_test_pred = np.mean(meta_test_pred_folds, axis=0)

    # submission向けに0-1へ（AUCには不要、本番提出の見た目調整）
    meta_test_pred_01 = (meta_test_pred - meta_test_pred.min()) / (meta_test_pred.max() - meta_test_pred.min() + 1e-12)

    print("\n[Meta Ridge Result]")
    print(f"feature_mode = {feature_mode}")
    print(f"models       = {model_order}")
    print(f"OOF AUC      = {meta_oof_auc:.6f}")
    print("fold alpha   =", [x["best_alpha"] for x in fold_info])

    coef_df = pd.DataFrame({
        "feature": feat_names,
        # 参考として最後のfoldの係数だけではなく、全fold平均係数にしたい場合は追加実装
    })

    return {
        "meta_oof_pred": meta_oof_pred,
        "meta_test_pred": meta_test_pred,
        "meta_test_pred_01": meta_test_pred_01,
        "meta_oof_auc": meta_oof_auc,
        "fold_info": pd.DataFrame(fold_info),
        "feature_names": feat_names,
    }

# 利用モデルを選ぶ（PBを見ながら）
oof_dict = {
    "cat": oof_cat,
    "real": oof_real,
}
test_dict = {
    "cat": test_cat,
    "real": test_real,
}

if "oof_xgb" in globals():
    oof_dict["xgb"] = oof_xgb
    test_dict["xgb"] = test_xgb

if "oof_cat_alt" in globals():
    oof_dict["cat_alt"] = oof_cat_alt
    test_dict["cat_alt"] = test_cat_alt

# 使うモデル順
model_order = [m for m in ["cat", "real", "cat_alt", "xgb"] if m in oof_dict]

# 3種類比較して一番安定なものを採用
results_meta = {}
for feature_mode in ["rank", "prob", "rank+prob"]:
    print("\n" + "="*80)
    print("feature_mode:", feature_mode)

    res = fit_meta_ridge_oof_and_test(
        y=y_comp.values,
        oof_dict=oof_dict,
        test_dict=test_dict,
        model_order=model_order,
        alpha_grid=(0.01, 0.1, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0),
        feature_mode=feature_mode,
        n_splits=5,
        seed=42,
    )
    results_meta[feature_mode] = res

# 最良を選ぶ（まずはCV基準）
best_mode = max(results_meta.keys(), key=lambda k: results_meta[k]["meta_oof_auc"])
print("\nBEST META MODE:", best_mode, results_meta[best_mode]["meta_oof_auc"])



feature_mode: rank
[Meta Ridge][Fold 1] AUC=0.956126, alpha=10.0
[Meta Ridge][Fold 2] AUC=0.954976, alpha=3.0
[Meta Ridge][Fold 3] AUC=0.955793, alpha=3.0
[Meta Ridge][Fold 4] AUC=0.955404, alpha=10.0
[Meta Ridge][Fold 5] AUC=0.956264, alpha=30.0

[Meta Ridge Result]
feature_mode = rank
models       = ['cat', 'real', 'cat_alt', 'xgb']
OOF AUC      = 0.955707
fold alpha   = [10.0, 3.0, 3.0, 10.0, 30.0]

feature_mode: prob
[Meta Ridge][Fold 1] AUC=0.956151, alpha=0.01
[Meta Ridge][Fold 2] AUC=0.954990, alpha=0.1
[Meta Ridge][Fold 3] AUC=0.955810, alpha=3.0
[Meta Ridge][Fold 4] AUC=0.955455, alpha=0.1
[Meta Ridge][Fold 5] AUC=0.956278, alpha=3.0

[Meta Ridge Result]
feature_mode = prob
models       = ['cat', 'real', 'cat_alt', 'xgb']
OOF AUC      = 0.955717
fold alpha   = [0.01, 0.1, 3.0, 0.1, 3.0]

feature_mode: rank+prob
[Meta Ridge][Fold 1] AUC=0.956147, alpha=30.0
[Meta Ridge][Fold 2] AUC=0.954983, alpha=3.0
[Meta Ridge][Fold 3] AUC=0.955811, alpha=30.0
[Meta Ridge][Fold 4] AUC=0.955

### Comparison


In [25]:
# ---- 比較（既存ブレンド vs メタ学習）----
oof_simple = 0.9 * oof_cat + 0.1 * oof_real
auc_simple = roc_auc_score(y_comp, oof_simple)

rank_blend_ref = np.zeros(len(y_comp), dtype="float64")
for m in model_order:
    rank_blend_ref += rank01(oof_dict[m]) / len(model_order)
auc_rank_ref = roc_auc_score(y_comp, rank_blend_ref)

auc_meta = results_meta[best_mode]["meta_oof_auc"]

print(f"[Cat only]         OOF AUC = {roc_auc_score(y_comp, oof_cat):.6f}")
print(f"[RealMLP only]     OOF AUC = {roc_auc_score(y_comp, oof_real):.6f}")
if "oof_xgb" in globals():
    print(f"[XGB only]         OOF AUC = {roc_auc_score(y_comp, oof_xgb):.6f}")
if "oof_cat_alt" in globals():
    print(f"[Cat_ALT only]     OOF AUC = {roc_auc_score(y_comp, oof_cat_alt):.6f}")

print(f"[Simple 0.9/0.1]   OOF AUC = {auc_simple:.6f}")
print(f"[Rank mean]        OOF AUC = {auc_rank_ref:.6f}")
print(f"[Meta Ridge]       OOF AUC = {auc_meta:.6f}")


[Cat only]         OOF AUC = 0.955610
[RealMLP only]     OOF AUC = 0.955657
[XGB only]         OOF AUC = 0.955220
[Cat_ALT only]     OOF AUC = 0.955639
[Simple 0.9/0.1]   OOF AUC = 0.955644
[Rank mean]        OOF AUC = 0.955694
[Meta Ridge]       OOF AUC = 0.955717


### Submission (Meta Ridge)


In [26]:
best_res = results_meta[best_mode]

# OOF保存
meta_oof_df = pd.DataFrame({
    "id": comp_ids,
    "y_true": y_comp.values,
    "y_pred": best_res["meta_oof_pred"],
})
meta_oof_df.to_csv("044_meta_ridge_rankprob_oof.csv", index=False)

# 提出保存
submission = pd.DataFrame({
    "id": test_ids,
    "Heart Disease": best_res["meta_test_pred_01"],
})
submission.to_csv("submission_meta_ridge_rankprob.csv", index=False)

print("Saved:")
print("  044_meta_ridge_rankprob_oof.csv")
print("  submission_meta_ridge_rankprob.csv")
print("Best mode:", best_mode)
print("OOF AUC :", best_res["meta_oof_auc"])


Saved:
  044_meta_ridge_rankprob_oof.csv
  submission_meta_ridge_rankprob.csv
Best mode: prob
OOF AUC : 0.9557170083468736
